# Uncertainty Analysis: Monte Carlo vs Chance-Constrained Optimization

This notebook provides a comprehensive comparison between Monte Carlo and Chance-Constrained (CC) optimization approaches for the ammonia production case study.

## Analysis Components:
1. **CC-Pareto Optimization**: Solving the chance-constrained Pareto problem
2. **Monte Carlo Analysis**: Two uncertainty approaches (custom strategies vs fitted normal)
3. **Comparative Visualization**: Progressive comparison plots
4. **Choice Analysis**: Technology pathway analysis by risk ranges
5. **Analytical Impact Distributions**: Deriving impact distributions directly from CC metadata

All functions are modularized in `a1_uncertainty_mc_utils.py` and `a1_uncertainty_mc_plots.py`.\n\n> **Note on imports:** Uncertainty features are exposed through `pulpo.pulpo_unc.PulpoOptimizerUnc` (a subclass of the base `pulpo.pulpo.PulpoOptimizer`) and require the `uncertainty` extras: `pip install \"pulpo-dev[uncertainty]\"`. Basic Monte Carlo over raw Brightway uncertainty (`solve_MC`) remains available on the base `PulpoOptimizer` with no extra dependencies.

In [ ]:
# Configuration parameters
FORCE_RECALCULATION = False
N_MC_ITERATIONS = 1500
RANDOM_SEED = 666
SOLVER_NAME = "gurobi"
UNCERTAINTY_CUTOFF = 0.000001
N_TOP_PROCESSES = 19

# File paths
RESULTS_DIR = "data/results"
CC_PARETO_FILE = f"{RESULTS_DIR}/cc_pareto_results.pkl"
MC_STRATEGIES_FILE = f"{RESULTS_DIR}/mc_uncertainty_strategies.pkl"
MC_NORMAL_FILE = f"{RESULTS_DIR}/mc_fitted_normal.pkl"

print("✓ Configuration loaded successfully!")

In [ ]:
# Enable automatic reloading of modified modules
%load_ext autoreload
%autoreload 2

# Import required libraries
import os
import numpy as np
import pandas as pd

# Ensure results directory exists
os.makedirs(RESULTS_DIR, exist_ok=True)

# Import our custom modules
import notebook_utils.a1_uncertainty_mc_utils as unc_utils
import notebook_utils.a1_uncertainty_mc_plots as unc_plots

# PULPO imports
from pulpo import pulpo_unc
from pulpo.utils.uncertainty import processor

print("✓ All libraries imported successfully!")
print("✓ Autoreload enabled - modules will update automatically when modified!")

### Import Libraries and Initialize Modules

Import necessary libraries and enable autoreload to automatically update custom modules when they are modified. This is useful during development and debugging.

## 1. PULPO Worker Initialization

Initialize the PULPO worker with the ammonia case study (assuming databases are already installed).

### Initialize PULPO Worker

The PULPO optimizer is initialized with:
- **Project**: ammonia_last - the ammonia production case study
- **Database**: ecoinvent 3.10 cutoff database + custom ammonia database
- **Method**: IPCC 2013 climate change impact assessment with uncertainty
- The intervention matrix links process activities to environmental flows

In [ ]:
# Project configuration
PROJECT = "ammonia_final"
DATABASE = ["ecoinvent-3.10-cutoff", "ammonia"]
METHOD = "('IPCC 2013', 'climate change', 'global warming potential (GWP100)', 'uncertain')"
DIRECTORY = "develop_tests"

# Initialize PULPO worker
pulpo_worker = pulpo_unc.PulpoOptimizerUnc(PROJECT, DATABASE, METHOD, DIRECTORY)
pulpo_worker.intervention_matrix = "ecoinvent-3.10-biosphere"
pulpo_worker.get_lci_data()

print("\n✓ PULPO worker initialized!")
print(f"  Project: {PROJECT}")
print(f"  Database: {DATABASE}")
print(f"  Method: {METHOD}")

### Define Problem Structure

The ammonia problem is defined with:
- **Choices**: Technology options for each process category (hydrogen production, methane sources, heat generation, etc.)
- **Demand**: Functional unit (1e7 kg ammonia per year)
- **Constraints**: Upper bounds on certain technologies to ensure realistic scenarios

In [ ]:
# Define ammonia problem (choices, demand, upper bounds)
choices, demand = unc_utils.define_ammonia_problem(pulpo_worker)

# Instantiate the problem
pulpo_worker.instantiate(demand=demand, choices=choices)

print("\n✓ Ammonia problem defined and instantiated!")
print(f"  Choices: {len(choices)} technology categories")
print(f"  Demand: {list(demand.values())[0]:.2e} kg/yr ammonia")

## 2. Chance-Constrained Pareto Optimization

In [ ]:
# Run or load CC-Pareto optimization
if unc_utils.check_file_exists(CC_PARETO_FILE, FORCE_RECALCULATION):
    print(f"Loading existing CC-Pareto results from {CC_PARETO_FILE}...")
    cc_pareto_results = unc_utils.load_results(CC_PARETO_FILE)
else:
    print("Running CC-Pareto optimization...")
    
    # Get uncertainty strategies (base iteration)
    unc_strategies = unc_utils.get_uncertainty_strategies(METHOD)
    
    # Import and filter uncertainty data
    pulpo_worker.import_and_filter_uncertainty_data(
        cutoff=UNCERTAINTY_CUTOFF,
        scaling_vector_strategy='constructed_demand',
        plot_results=False,
        plot_n_top_processes=N_TOP_PROCESSES
    )
    
    # Apply all uncertainty strategies including expert knowledge refinements
    unc_utils.apply_all_expert_strategies(pulpo_worker, unc_strategies)
    
    # Create CC formulation
    normal_metadata_env_cost, normal_metadata_var_bounds = pulpo_worker.create_CC_formulation(
        CC_env_cost=True, CC_var_bounds=['upper_limit']
    )
    
    # Solve CC-Pareto problem
    lambda_epsilon_array = np.arange(0.02, 1, 0.02)
    results_CC = pulpo_worker.solve_CC_problem(
        lambda_epsilon_array, normal_metadata_env_cost, normal_metadata_var_bounds, 
        solver_name=SOLVER_NAME
    )
    
    # Package and save results
    cc_pareto_results = {
        'results_CC': results_CC,
        'lambda_epsilon_array': lambda_epsilon_array,
        'normal_metadata_env_cost': normal_metadata_env_cost,
        'normal_metadata_var_bounds': normal_metadata_var_bounds,
        'uncertainty_data': pulpo_worker.uncertainty_data
    }
    unc_utils.save_results(cc_pareto_results, CC_PARETO_FILE)

print(f"\n✓ CC-Pareto optimization completed! Generated {len(cc_pareto_results['results_CC'])} Pareto points")

## 3. Monte Carlo Analysis (Two Approaches)

In [ ]:
# Monte Carlo Analysis 1: Custom Uncertainty Strategies
if unc_utils.check_file_exists(MC_STRATEGIES_FILE, FORCE_RECALCULATION):
    print(f"Loading existing MC results (strategies) from {MC_STRATEGIES_FILE}...")
    mc_strategies_results = unc_utils.load_results(MC_STRATEGIES_FILE)
else:
    print(f"Running Monte Carlo analysis with custom strategies ({N_MC_ITERATIONS} iterations)...")
    
    # Re-define problem for MC analysis
    choices, demand = unc_utils.define_ammonia_problem(pulpo_worker)
    
    # Get base uncertainty strategies
    unc_strategies = unc_utils.get_uncertainty_strategies(METHOD)
    
    # Import and filter uncertainty data
    pulpo_worker.import_and_filter_uncertainty_data(
        cutoff=UNCERTAINTY_CUTOFF, scaling_vector_strategy='constructed_demand',
        plot_results=False, plot_n_top_processes=N_TOP_PROCESSES
    )
    
    # Apply all uncertainty strategies including expert knowledge refinements
    unc_utils.apply_all_expert_strategies(pulpo_worker, unc_strategies)
    
    # Run Monte Carlo simulation
    mc_strategies_results = pulpo_worker.run_mc_from_uncertainty(
        n_samples=N_MC_ITERATIONS, seed=RANDOM_SEED, solver_name=SOLVER_NAME, options=None
    )
    unc_utils.save_results(mc_strategies_results, MC_STRATEGIES_FILE)

print("\n✓ MC analysis with custom strategies completed!")
analysis_strategies = unc_utils.analyze_MC_results(mc_strategies_results, show_plot=True)

In [ ]:
# Monte Carlo Analysis 2: Fitted Normal Distributions
if unc_utils.check_file_exists(MC_NORMAL_FILE, FORCE_RECALCULATION):
    print(f"Loading existing MC results (normal) from {MC_NORMAL_FILE}...")
    mc_normal_results = unc_utils.load_results(MC_NORMAL_FILE)
else:
    print(f"Running Monte Carlo analysis with fitted normal distributions ({N_MC_ITERATIONS} iterations)...")
    
    # Re-define problem for MC analysis
    choices, demand = unc_utils.define_ammonia_problem(pulpo_worker)
    
    # Get base uncertainty strategies
    unc_strategies = unc_utils.get_uncertainty_strategies(METHOD)
    
    # Import and filter uncertainty data
    pulpo_worker.import_and_filter_uncertainty_data(
        cutoff=UNCERTAINTY_CUTOFF, scaling_vector_strategy='constructed_demand',
        plot_results=False, plot_n_top_processes=N_TOP_PROCESSES
    )
    
    # Apply all uncertainty strategies including expert knowledge refinements
    unc_utils.apply_all_expert_strategies(pulpo_worker, unc_strategies)
    
    # Transform distributions to normal
    pulpo_worker.uncertainty_data = processor.transform_to_normal(
        pulpo_worker.uncertainty_data, sample_size=100, plot_distribution=False
    )
    
    # Run Monte Carlo simulation
    mc_normal_results = pulpo_worker.run_mc_from_uncertainty(
        n_samples=N_MC_ITERATIONS, seed=RANDOM_SEED, solver_name=SOLVER_NAME, options=None
    )
    unc_utils.save_results(mc_normal_results, MC_NORMAL_FILE)

print("\n✓ MC analysis with normal distributions completed!")
analysis_normal = unc_utils.analyze_MC_results(mc_normal_results, show_plot=True)

## 4. Comparative Analysis and Visualization

In [ ]:
# Create comparative summary table
comparison_df = unc_plots.create_summary_table(analysis_strategies, analysis_normal)

In [ ]:
# Generate progressive comparative plots
unc_plots.plot_comparative_mc_analysis(
    analysis_strategies, analysis_normal, 
    cc_pareto_results, RESULTS_DIR
)

## 5. Choice Analysis within Risk Ranges

In [ ]:
# Analyze technology choices across different risk ranges
print("=" * 80)
print("CHOICE ANALYSIS FOR DIFFERENT RISK SCENARIOS")
print("=" * 80)

risk_ranges = [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 0.95), (0.95, 1.0)]

risk_analysis_results = {
    f"{int(a*100)}%-{int(b*100)}%": res
    for (a, b) in risk_ranges
    if (res := unc_utils.analyze_choices_in_risk_range(
        mc_normal_results, analysis_normal, a, b, f"{int(a*100)}%-{int(b*100)}%"
    ))
}

print("\n" + "=" * 80)
print("CHOICE ANALYSIS COMPLETE")
print("=" * 80)

In [ ]:
# Visualize choice analysis
unc_plots.plot_choice_analysis(
    risk_analysis_results, mc_normal_results, 
    f"{RESULTS_DIR}/choice_analysis_by_risk.png"
)

## 6. Analytical Impact Distribution from CC Metadata

Instead of Monte Carlo sampling, we can analytically compute the impact distribution using the mean (`loc`) and standard deviation (`scale`) from `normal_metadata_env_cost` combined with the scaling vector.

For a weighted sum of independent normal distributions:
- If $X_i \sim N(\mu_i, \sigma_i^2)$ with scaling factors $s_i$
- Then $\sum s_i X_i \sim N\left(\sum s_i \mu_i, \sum s_i^2 \sigma_i^2\right)$

This gives us the analytical distribution of total environmental impact for each confidence level.

In [ ]:
# Define confidence levels for analytical distribution analysis
# Use round() to avoid floating point precision issues like 0.9400000000000001
CC_CONFIDENCE_LEVELS = [round(x, 2) for x in [0.50, 0.6, 0.7, 0.80, 0.9, 0.94, 0.98]]
print(f"Confidence levels: {CC_CONFIDENCE_LEVELS}")

In [ ]:
# Compute analytical distributions for the confidence levels
print("Computing analytical impact distributions from CC metadata...")
print("=" * 80)
analytical_distributions = unc_utils.compute_analytical_distributions_for_all_lambdas(
    cc_pareto_results, CC_CONFIDENCE_LEVELS
)
print("=" * 80)
print(f"\n✓ Analytical distributions computed for {len(analytical_distributions)} confidence levels")

In [ ]:
# Define thresholds (in kg CO2-eq)
THRESHOLDS = {
    "PB": 3.70144e9,   # 3.70144 Mt CO2-eq (planetary boundary)
    "18 Mt": 18e9      # 20 Mt CO2-eq
}

# Visualize analytical impact distributions (PDFs and CDFs) with thresholds
fig_analytical = unc_plots.plot_analytical_distributions(
    analytical_distributions, 
    save_path=f"{RESULTS_DIR}/analytical_impact_distributions.png",
    thresholds=THRESHOLDS
)

In [ ]:
# Create summary statistics table
print("\n" + "="*80)
print("ANALYTICAL IMPACT DISTRIBUTION SUMMARY")
print("="*80)

summary_data = []
for lambda_val, params in analytical_distributions.items():
    summary_data.append({
        'Confidence Level (λ)': lambda_val,
        'Mean Impact': f"{params['mean']:.2e}",
        'Std Dev': f"{params['std']:.2e}",
        'CV (%)': f"{params['std']/params['mean']*100:.1f}",
        'Variance': f"{params['variance']:.2e}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("="*80)

print(f"\n✓ Analytical impact distribution analysis completed!")
print(f"✓ Visualization saved to: {RESULTS_DIR}/analytical_impact_distributions.png")